# Phase 2: Tamil Data Expansion Pipeline

**Goal**: Expand the Tamil training corpus beyond Wikipedia with free HuggingFace datasets.

**Sources (all free)**:
- `cc100` (lang=ta) — web-crawled Tamil
- `ai4bharat/IndicCorp` — curated Indic corpus, Tamil shard
- `oscar-corpus/OSCAR-2301` — deduplicated Common Crawl Tamil

**Output**: Combined, deduplicated, quality-filtered dataset pushed to HF

**Hardware**: Colab 40GB recommended (large downloads)

In [ ]:
# ── Install ────────────────────────────────────────────────────────────────────
import subprocess, sys

for pkg in ["datasets", "datasketch", "langdetect", "tqdm"]:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", pkg])

print("Ready.")

In [ ]:
# ── Configuration ──────────────────────────────────────────────────────────────
# Adjust MAX_SAMPLES_PER_SOURCE to control dataset size.
# None = download everything (slow, large storage required)

MAX_SAMPLES_PER_SOURCE = 500_000   # 500K per source; total ~2M after mixing

HF_OUTPUT_REPO = "wickkiey/tamil-corpus-expanded"
HF_TOKEN       = None   # or set your token string

MIN_LENGTH      = 50       # minimum char length per sample
MIN_TAMIL_RATIO = 0.80     # minimum fraction of Tamil Unicode characters

# Mix ratios (must sum to 1.0)
MIX_RATIOS = {
    "wikipedia": 0.40,
    "indic_corp": 0.30,
    "cc100":      0.20,
    "oscar":      0.10,
}

FINAL_SIZE = 1_000_000   # ~1M samples total after mixing

print("Config loaded.")
print(f"Mix ratios: {MIX_RATIOS}")

In [ ]:
# ── Quality filter function ────────────────────────────────────────────────────
import unicodedata

TAMIL_UNICODE_START = 0x0B80
TAMIL_UNICODE_END   = 0x0BFF

def tamil_char_ratio(text: str) -> float:
    """Fraction of characters in the Tamil Unicode block."""
    if not text:
        return 0.0
    tamil_chars = sum(
        1 for c in text
        if TAMIL_UNICODE_START <= ord(c) <= TAMIL_UNICODE_END
    )
    return tamil_chars / len(text)

def is_quality(text: str) -> bool:
    """Return True if the sample passes quality filters."""
    if not text or len(text.strip()) < MIN_LENGTH:
        return False
    if tamil_char_ratio(text) < MIN_TAMIL_RATIO:
        return False
    # Remove obvious template noise (MediaWiki artifacts)
    if text.strip().startswith(("{{" , "[[" , "==")):
        return False
    return True

# Quick test
assert is_quality("தமிழ்நாடு இந்தியாவின் தென் மாநிலங்களில் ஒன்றாகும்.") == True
assert is_quality("hello world") == False   # no Tamil chars
assert is_quality("") == False
print("Quality filter OK.")

In [ ]:
# ── Load Wikipedia (already have locally) ─────────────────────────────────────
from datasets import load_dataset

print("Loading Tamil Wikipedia (chunked) from HF...")
wiki_ds = load_dataset("wickkiey/tamil-wikipedia-markdown", split="train")

# Normalize column name to 'text'
if "text" not in wiki_ds.column_names:
    wiki_ds = wiki_ds.rename_column(wiki_ds.column_names[0], "text")

wiki_ds = wiki_ds.filter(lambda x: is_quality(x["text"]), desc="Filtering Wikipedia")
wiki_ds = wiki_ds.select_columns(["text"])

n_wiki = int(FINAL_SIZE * MIX_RATIOS["wikipedia"])
if len(wiki_ds) > n_wiki:
    wiki_ds = wiki_ds.select(range(n_wiki))

print(f"Wikipedia samples : {len(wiki_ds):,}")

In [ ]:
# ── Load CC-100 Tamil ──────────────────────────────────────────────────────────
print("Loading CC-100 Tamil (streaming to avoid full download)...")
cc100_stream = load_dataset(
    "cc100",
    lang      = "ta",
    split     = "train",
    streaming = True,    # stream to avoid downloading all 17GB
)

n_cc100 = int(FINAL_SIZE * MIX_RATIOS["cc100"])
cc100_samples = []
for sample in cc100_stream:
    if is_quality(sample["text"]):
        cc100_samples.append({"text": sample["text"]})
    if len(cc100_samples) >= n_cc100:
        break

from datasets import Dataset
cc100_ds = Dataset.from_list(cc100_samples)
print(f"CC-100 samples    : {len(cc100_ds):,}")

In [ ]:
# ── Load IndicCorp Tamil ───────────────────────────────────────────────────────
print("Loading AI4Bharat IndicCorp Tamil (streaming)...")
try:
    indic_stream = load_dataset(
        "ai4bharat/IndicCorp",
        "ta",
        split     = "train",
        streaming = True,
    )

    n_indic = int(FINAL_SIZE * MIX_RATIOS["indic_corp"])
    indic_samples = []
    for sample in indic_stream:
        text = sample.get("text") or sample.get("sentence") or ""
        if is_quality(text):
            indic_samples.append({"text": text})
        if len(indic_samples) >= n_indic:
            break

    indic_ds = Dataset.from_list(indic_samples)
    print(f"IndicCorp samples : {len(indic_ds):,}")
except Exception as e:
    print(f"IndicCorp load failed: {e}")
    print("Falling back to empty dataset. Add IndicCorp manually if needed.")
    indic_ds = Dataset.from_list([])

In [ ]:
# ── Load OSCAR Tamil ───────────────────────────────────────────────────────────
print("Loading OSCAR 2301 Tamil (streaming)...")
try:
    oscar_stream = load_dataset(
        "oscar-corpus/OSCAR-2301",
        language  = "ta",
        split     = "train",
        streaming = True,
        trust_remote_code = True,
    )

    n_oscar = int(FINAL_SIZE * MIX_RATIOS["oscar"])
    oscar_samples = []
    for sample in oscar_stream:
        text = sample.get("text", "")
        if is_quality(text):
            oscar_samples.append({"text": text})
        if len(oscar_samples) >= n_oscar:
            break

    oscar_ds = Dataset.from_list(oscar_samples)
    print(f"OSCAR samples     : {len(oscar_ds):,}")
except Exception as e:
    print(f"OSCAR load failed: {e}")
    oscar_ds = Dataset.from_list([])

In [ ]:
# ── Combine all sources ────────────────────────────────────────────────────────
from datasets import concatenate_datasets

all_parts = [ds for ds in [wiki_ds, indic_ds, cc100_ds, oscar_ds] if len(ds) > 0]
combined = concatenate_datasets(all_parts)
combined = combined.shuffle(seed=42)

print(f"\nCombined dataset  : {len(combined):,} samples")
print(f"Estimated size    : {sum(len(x['text']) for x in combined) / 1e9:.2f} GB (chars)")

In [ ]:
# ── MinHash deduplication ──────────────────────────────────────────────────────
# Remove near-duplicate texts using MinHash Locality-Sensitive Hashing.
# This prevents Wikipedia content from appearing 2x (once from wiki, once from OSCAR).
from datasketch import MinHash, MinHashLSH
from tqdm import tqdm

def text_to_minhash(text: str, num_perm: int = 128) -> MinHash:
    m = MinHash(num_perm=num_perm)
    for word in text.lower().split():
        m.update(word.encode("utf-8"))
    return m

SIMILARITY_THRESHOLD = 0.85  # texts with >85% similarity are duplicates
NUM_PERM = 128

lsh = MinHashLSH(threshold=SIMILARITY_THRESHOLD, num_perm=NUM_PERM)
keep_indices = []

print(f"Deduplicating {len(combined):,} samples (threshold={SIMILARITY_THRESHOLD})...")
for i, sample in tqdm(enumerate(combined), total=len(combined)):
    mh = text_to_minhash(sample["text"], NUM_PERM)
    key = str(i)
    if not lsh.query(mh):        # no similar document found
        lsh.insert(key, mh)
        keep_indices.append(i)

deduped = combined.select(keep_indices)
removed = len(combined) - len(deduped)
print(f"\nBefore dedup : {len(combined):,}")
print(f"After dedup  : {len(deduped):,}  (removed {removed:,} near-duplicates)")

In [ ]:
# ── Dataset statistics ─────────────────────────────────────────────────────────
import statistics

lengths = [len(x["text"]) for x in deduped]
print(f"Total samples   : {len(deduped):,}")
print(f"Total chars     : {sum(lengths):,}")
print(f"Avg length      : {statistics.mean(lengths):.0f} chars")
print(f"Median length   : {statistics.median(lengths):.0f} chars")
print(f"Min length      : {min(lengths)} chars")
print(f"Max length      : {max(lengths)} chars")

In [ ]:
# ── Push to HuggingFace ────────────────────────────────────────────────────────
PUSH_TO_HF = True

if PUSH_TO_HF:
    print(f"Pushing to HF: {HF_OUTPUT_REPO}")
    deduped.push_to_hub(
        HF_OUTPUT_REPO,
        token          = HF_TOKEN,
        commit_message = "Tamil expanded corpus: Wikipedia + IndicCorp + CC-100 + OSCAR (deduplicated)"
    )
    print(f"Done: https://huggingface.co/datasets/{HF_OUTPUT_REPO}")
else:
    deduped.save_to_disk("tamil_expanded_corpus")
    print("Saved locally to: tamil_expanded_corpus/")

## Next Step

Use `wickkiey/tamil-corpus-expanded` as the dataset in a **second CPT run** (`01_cpt_llama31_8b.ipynb`) with `HF_DATASET = "wickkiey/tamil-corpus-expanded"` — this will further improve the model's Tamil fluency with broader vocabulary coverage.